# Download Yolov8 library 

In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.8/993.8 kB 22.2 MB/s eta 0:00:00


# Import libraries 

In [2]:
import os
from pathlib import Path
import yaml
import shutil
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import glob
import json
import csv
from pathlib import Path
import cv2
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Prepare YOLO Dataset Format

In [ ]:
def setup_kaggle_dataset(competition_dir):

    # Define paths of data , images and annotations
    train_img_dir = os.path.join(competition_dir, '/kaggle/input/pulmonary-nodule/train/jpg')
    train_anno_dir = os.path.join(competition_dir, '/kaggle/input/pulmonary-nodule/train/anno')

    # Create YOLO dataset directory structure
    yolo_dir = Path('yolo_dataset')
    for subdir in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        (yolo_dir / subdir).mkdir(parents=True, exist_ok=True)
    

    # Get all training images
    train_images = [f for f in os.listdir(train_img_dir) if f.endswith('.jpg')]
    

    # Split into train and validation sets
    train_files, val_files = train_test_split(train_images, test_size=0.2, random_state=42)
    

    # Process files validation and train files 
    def process_files(files, subset='train'):
        for img_file in files:
            # Copy image
            src_img = Path(train_img_dir) / img_file
            dst_img = yolo_dir / subset / 'images' / img_file
            shutil.copy(src_img, dst_img)
            
            # Convert annotation
            base_name = img_file.rsplit('.', 1)[0]
            xml_path = Path(train_anno_dir) / f"{base_name}.xml"
            if xml_path.exists():

                # Get image dimensions
                img = cv2.imread(str(src_img))
                img_height, img_width = img.shape[:2]
                

                # Convert XML to YOLO format
                txt_path = yolo_dir / subset / 'labels' / f"{base_name}.txt"
                tree = ET.parse(xml_path)
                root = tree.getroot()

                # extract bounding box coordinates
                with open(txt_path, 'w') as f:
                    for obj in root.findall('object'):
                        if obj.find('name').text != 'nodule':
                            continue
                        
                        bbox = obj.find('bndbox')
                        xmin = float(bbox.find('xmin').text)
                        ymin = float(bbox.find('ymin').text)
                        xmax = float(bbox.find('xmax').text)
                        ymax = float(bbox.find('ymax').text)
                        
                        # Convert to YOLO format
                        x_center = (xmin + xmax) / (2 * img_width)
                        y_center = (ymin + ymax) / (2 * img_height)
                        width = (xmax - xmin) / img_width
                        height = (ymax - ymin) / img_height
                        
                        f.write(f"0 {x_center} {y_center} {width} {height}\n")
    
    # Process train and validation sets
    process_files(train_files, 'train')
    process_files(val_files, 'val')
    

    # Create dataset YAML
    dataset_config = {
        'path': str(yolo_dir.absolute()),
        'train': 'train/images',
        'val': 'val/images',
        'names': {
            0: 'nodule'
        }
    }
    
    # Save dataset YAML
    yaml_path = yolo_dir / 'dataset.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(dataset_config, f)
    
    return str(yaml_path)

# define function of training

In [ ]:
def train_model():
    
    #Train YOLOv8 model 

    # Set up dataset
    dataset_yaml = setup_kaggle_dataset('.')  # Current directory in Kaggle
    

    # Initialize model
    model = YOLO('yolov8s.pt')  #small model
    
    training_args = {
    'data': dataset_yaml,
    'epochs': 50,            # i have tried 100 nd 32 and 50 , best was 50
    'imgsz': 1024,            # Keep resolution high for small nodules
    'batch': 8,
    'patience': 20,           # Early stopping 
    'optimizer': 'AdamW',
    'lr0': 0.0005,            # Lower learning rate for stability
    'weight_decay': 0.0005,  

    # Data augmentation techniques  
    'hsv_h': 0.015,  # Hue augmentation  
    'hsv_s': 0.7,    # Saturation augmentation  
    'hsv_v': 0.4,    # Brightness augmentation  
    'fliplr': 0.5,   # Horizontal flip  
    'flipud': 0.3,   # Vertical flip  
    'mosaic': 1.0,   # Enable mosaic augmentation  
    'mixup': 0.1,    # Enable mixup augmentation  

    'overlap_mask': True,
    'val': True,
    'save': True,
   }

    # Train the model
    results = model.train(**training_args)
    return model, results

In [5]:
print("Starting training...")
model, results = train_model()
print("Training completed!")    

# Save the model
model.save('best_nodule_detector.pt')

Starting training...


100%|██████████| 21.5M/21.5M [00:00<00:00, 220MB/s]


Ultralytics 8.3.102 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=yolo_dataset/dataset.yaml, epochs=50, time=None, patience=20, batch=8, imgsz=1024, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes

100%|██████████| 755k/755k [00:00<00:00, 26.1MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

...


50 epochs completed in 0.765 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 22.6MB
Optimizer stripped from runs/detect/train/weights/best.pt, 22.6MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.102 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 19/19 [00:06<00:00,  2.97it/s]


                   all        300       1145      0.802       0.79      0.871       0.53


/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.5ms preprocess, 12.2ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs/detect/train
Training completed!


In [ ]:
import csv
import json
import glob
import os
import cv2
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

def compute_iou(box1, box2):
    """
    Compute Intersection over Union (IoU) between two bounding boxes.
    """
    x1 = max(box1["xmin"], box2["xmin"])
    y1 = max(box1["ymin"], box2["ymin"])
    x2 = min(box1["xmax"], box2["xmax"])
    y2 = min(box1["ymax"], box2["ymax"])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1["xmax"] - box1["xmin"]) * (box1["ymax"] - box1["ymin"])
    box2_area = (box2["xmax"] - box2["xmin"]) * (box2["ymax"] - box2["ymin"])
    
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area else 0

def apply_nms(predictions, iou_thresh=0.5):
    """
    Apply Non-Maximum Suppression (NMS) to remove overlapping bounding boxes.
    """
    filtered_preds = []
    for i, row in predictions.iterrows():
        keep = True
        for selected in filtered_preds:
            iou = compute_iou(row, selected)
            if iou > iou_thresh:
                keep = False
                break
        if keep:
            filtered_preds.append(row.to_dict())  # Convert row to dict before storing
    return filtered_preds


# this project was a challenge in kaggle and submission was to be done in csv format 
#this function generates the submission file in the required format using the trained model

def generate_submission(test_img_dir, output_csv='submission.csv', conf_threshold=0.15 ,  iou_thresh = 0.5 ):
    """
    Generate submission from saved YOLOv8 model with adjusted confidence threshold and NMS IoU.
    """
    # Load trained model
    model = YOLO('/kaggle/working/best_nodule_detector.pt')
    
    # Get test images
    test_images = sorted(glob.glob(os.path.join(test_img_dir, '*.jpg')))
    
    results = []
    for img_path in test_images:
        filename = Path(img_path).stem
        img = cv2.imread(img_path)
        height, width = img.shape[:2]

        # Run prediction with updated settings
        predictions = model.predict(img_path, conf=conf_threshold, iou= iou_thresh, imgsz=1024, verbose=False)[0]

        # Extract boxes and scores
        boxes = predictions.boxes.data.cpu().numpy()
        
        objects = []
        for box in boxes:
            x1, y1, x2, y2, conf, class_id = box
            bbox = {
                "xmin": int(x1),
                "ymin": int(y1),
                "xmax": int(x2),
                "ymax": int(y2)
            }
            objects.append({
                "class": "nodule",
                "bbox": bbox
            })

        # Store results
        results.append({
            "filename": filename,
            "width": width,
            "height": height,
            "depth": 3,
            "objects": json.dumps(objects)
        })

    # Save submission CSV
    with open(output_csv, 'w', newline='') as csvfile:
        fieldnames = ['filename', 'width', 'height', 'depth', 'objects']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in results:
            writer.writerow(row)
    
    print(f"\nSubmission CSV saved as {output_csv}")



    # Some statistics about the submission file (result)
    total_images = len(results)
    images_with_nodules = sum(1 for row in results if len(json.loads(row['objects'])) > 0)
    total_nodules = sum(len(json.loads(row['objects'])) for row in results)

    print("\nPrediction Statistics:")
    print(f"Total images processed: {total_images}")
    print(f"Images with detected nodules: {images_with_nodules} ({images_with_nodules/total_images*100:.1f}%)")
    print(f"Total nodules detected: {total_nodules}")
    print(f"Average nodules per image: {total_nodules/total_images:.2f}")



In [ ]:
test_img_dir = '/kaggle/input/pulmonary-nodule/test/jpg'  
generate_submission(
    test_img_dir=test_img_dir,
    output_csv='SubmissionFinalAll.csv',
    conf_threshold=0.34,  # less means capture more nodules
    iou_thresh = 0.45  # Reduce false positives
)


Submission CSV saved as SubmissionFinalAll.csv

Prediction Statistics:
Total images processed: 516
Images with detected nodules: 490 (95.0%)
Total nodules detected: 1957
Average nodules per image: 3.79
